# Multi-Survey Light Curve Classifier — Results Explorer

Visualización interactiva de los resultados de `classify.py`.  
Carga un parquet/CSV de resultados y explora objeto a objeto.

**Plots disponibles por objeto:**
1. Distribución final de probabilidades (bar plot horizontal)
2. Consenso entre modelos para la clase ganadora (radar chart)
3. Heatmap de probabilidades por modelo y clase
4. Panel resumen numérico

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Configuración

In [ ]:
#Ruta al parquet de resultados de classify.py
RESULTS_PATH = './output/classify/classify_results.parquet'  # default: output de classify.py

#Modelos base activos en la clasificación que generó este archivo
ACTIVE_MODELS = ['A', 'B','C','D','E']

#Se usó metamodelo?
USE_META = True

#Orden de clases (debe coincidir con LABEL_ORDER del entrenamiento)
LABEL_ORDER = [
    'SNIa', 'SNIbc', 'SNII', 'SLSN',
    'QSO', 'AGN', 'Blazar',
    'YSO', 'CV/Nova',
    'LPV', 'E', 'DSCT', 'RRL', 'CEP', 'Periodic-Other'
]

#Ruta de guardado
SAVES_PATH = './output/figures/'

#Umbrales para advertencia de divergencia metamodelo/base
# Se activa cuando el metamodelo predice de forma espuria una clase que los modelos base no ven.
# DIV_THRESHOLD: diferencia mínima entre prob. final y prob. media de los modelos base para la clase ganadora
# BASE_MAX: probabilidad máxima que los modelos base dan a esa clase
# Si no se cumplen las dos condiciones, salta una advertencia y la clasificación puede ser equivocada (revisar modelos base)
DIV_THRESHOLD = 0.3
BASE_MAX      = 0.1

## 2. Carga de datos

In [ ]:
def load_results(path: str) -> pd.DataFrame:
    """Carga el parquet/csv de resultados y normaliza el índice a str."""
    if path.endswith('.csv'):
        df = pd.read_csv(path, index_col=0, dtype={0: str})
    else:
        df = pd.read_parquet(path)
    df.index = df.index.astype(str)
    return df

results = load_results(RESULTS_PATH)
print(f"Objetos cargados : {len(results)}")
print(f"Columnas         : {len(results.columns)}")
print(f"\nDistribución de clases predichas:")
print(results['predicted_class'].value_counts().to_string())

## 3. Funciones de visualización

In [ ]:
#Paleta de colores
# Transientes: rojos/naranjas; Estocásticos: azules/verdes; Periódicas: púrpuras
CLASS_COLORS = {
    'SNIa':           '#d7191c',
    'SNIbc':          '#e8601c',
    'SNII':           '#f4a736',
    'SLSN':           '#984ea3',
    'QSO':            '#377eb8',
    'AGN':            '#4dac26',
    'Blazar':         '#1a9641',
    'YSO':            '#74c476',
    'CV/Nova':        '#41ab5d',
    'LPV':            '#807dba',
    'E':              '#9e9ac8',
    'DSCT':           '#bcbddc',
    'RRL':            '#6baed6',
    'CEP':            '#3182bd',
    'Periodic-Other': '#969696',
}

#Colores representativos por modelo (stop medio de cada colormap)
# A=teal, B=amber, C=purple, D=green, E=pink, Meta=blue
# Consistente con bootstrap_confmat.py MODEL_CMAPS
MODEL_COLORS = {
    'A':    '#1D9E75',   # teal  (ZTF)
    'B':    '#BA7517',   # amber (LSST)
    'C':    '#9067B2',   # purple (ZTF+LSST+ATLAS cross-survey)
    'D':    '#639922',   # green (combined LC)
    'E':    '#D4537E',   # pink  (ATLAS)
    'Meta': '#378ADD',   # blue  (metamodelo)
}
MODEL_LABELS = {
    'A': 'A (ZTF)', 'B': 'B (LSST)', 'C': 'C (ZTF+LSST+ATLAS)',
    'D': 'D (Combined)', 'E': 'E (ATLAS)', 'Meta': 'Metamodelo',
}


def get_object_data(oid, df):
    row = df.loc[oid]
    final_probs = {cl: row[f'p_{cl}'] for cl in LABEL_ORDER if f'p_{cl}' in row.index}
    model_probs = {}
    for m in ACTIVE_MODELS:
        col = f'p_{LABEL_ORDER[0]}_model{m}'
        if col in row.index:
            probs = {cl: row[f'p_{cl}_model{m}'] for cl in LABEL_ORDER
                     if f'p_{cl}_model{m}' in row.index}
            if not all(np.isnan(v) for v in probs.values()):
                model_probs[m] = probs
    # Divergencia: cuánto eleva el metamodelo la clase ganadora respecto a los modelos base
    predicted = row['predicted_class']
    base_vals = [
        model_probs[m].get(predicted, 0.0)
        for m in model_probs
        if not np.isnan(model_probs[m].get(predicted, np.nan))
    ]
    p_base_mean      = float(np.mean(base_vals)) if base_vals else 0.0
    p_meta_final     = float(row[f'p_{predicted}']) if USE_META else p_base_mean
    meta_divergence  = p_meta_final - p_base_mean

    return {
        'oid':             oid,
        'predicted':       predicted,
        'second':          row.get('second_class', None),
        'max_prob':        row['max_prob'],
        'second_prob':     row.get('second_prob', np.nan),
        'margin':          row['max_prob'] - row.get('second_prob', 0),
        'entropy':         row['entropy'],
        'n_models':        int(row['n_models_used']),
        'final_probs':     final_probs,
        'model_probs':     model_probs,
        'p_base_mean':     p_base_mean,
        'meta_divergence': meta_divergence,
    }


#Plot 1: Distribución final de probabilidades
def plot_final_probs(ax, data):
    classes = LABEL_ORDER[::-1]
    probs   = [data['final_probs'].get(cl, 0.0) for cl in classes]
    win_col    = CLASS_COLORS.get(data['predicted'], '#555555')
    second_col = '#b2abd2'
    colors = [
        win_col    if cl == data['predicted']
        else (second_col if cl == data['second'] else '#cccccc')
        for cl in classes
    ]
    bars = ax.barh(classes, probs, color=colors, edgecolor='white',
                   linewidth=0.5, height=0.7)
    for bar, prob in zip(bars, probs):
        if prob > 0.01:
            ax.text(prob + 0.005, bar.get_y() + bar.get_height() / 2,
                    f'{prob:.3f}', va='center', ha='left', fontsize=8)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel('Probabilidad', fontsize=10)
    ax.set_title('Distribucion final de probabilidades',
                 fontsize=11, fontweight='bold', pad=10)
    ax.axvline(1 / 15, color='#888888', linestyle='--', linewidth=0.8,
               alpha=0.7, label='Prior uniforme (1/15)')
    ax.legend(fontsize=8, loc='lower right')


#Datos compartidos para plot 2
def _build_consensus_data(data):
    active = list(data['model_probs'].keys())
    wc = data['predicted']
    if USE_META:
        names   = active + ['Meta']
        labels  = [MODEL_LABELS[m] for m in active] + [MODEL_LABELS['Meta']]
        values  = [data['model_probs'][m].get(wc, 0.0) for m in active] + [data['max_prob']]
        cols    = [MODEL_COLORS[m] for m in active] + [MODEL_COLORS['Meta']]
    else:
        names   = active
        labels  = [MODEL_LABELS[m] for m in active]
        values  = [data['model_probs'][m].get(wc, 0.0) for m in active]
        cols    = [MODEL_COLORS[m] for m in active]
    return names, labels, values, cols


#Plot 2a: Radar chart (N >= 3)
def plot_model_consensus_radar(ax, data):
    wc = data['predicted']
    _, labels, values, colors_list = _build_consensus_data(data)
    N = len(labels)

    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    values_closed = values + [values[0]]
    angles_closed = angles + [angles[0]]

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.yaxis.grid(False)
    ax.xaxis.grid(False)
    ax.set_yticks([])
    ax.set_thetagrids(np.degrees(angles), labels, fontsize=8.5)
    for label, angle in zip(ax.get_xticklabels(), angles):
        x, y = label.get_position()
        text = label.get_text()
        if 'ZTF+LSST+ATLAS' in text:
            label.set_position((x, y - 0.25))
        elif 'A (ZTF)' in text or 'Combined' in text:
            label.set_position((x, y - 0.03))
        else:
            label.set_position((x, y - 0.12))
    # Anillos de referencia cada 0.2
    for r in np.arange(0.2, 1.01, 0.2):
        ax.plot(np.linspace(0, 2 * np.pi, 200), [r] * 200,
                color='#bbbbbb', linewidth=0.6, zorder=1)
        ax.text(0, r, f'{r:.1f}', ha='center', va='bottom',
                fontsize=7, color='#999999')

    # Prior uniforme
    prior = 1 / len(LABEL_ORDER)
    ax.plot(np.linspace(0, 2 * np.pi, 200), [prior] * 200,
            color='#888888', linestyle='--', linewidth=1.0, alpha=0.8, zorder=2)

    # Polígono
    win_col = CLASS_COLORS.get(wc, '#555555')
    ax.plot(angles_closed, values_closed, color=win_col, linewidth=2, zorder=3)
    ax.fill(angles_closed, values_closed, color=win_col, alpha=0.18, zorder=3)

    for angle, val, col in zip(angles, values, colors_list):
        ax.plot(angle, val, 'o', color=col, markersize=7, zorder=5,
                markeredgecolor='white', markeredgewidth=0.8)

    ax.set_ylim(0, 1)
    ax.set_title(f'Consenso: {wc}', fontsize=10, fontweight='bold', pad=10)


#Plot 2b: Bar chart fallback (N < 3)
def plot_model_consensus_bar(ax, data):
    wc = data['predicted']
    _, labels, values, colors_list = _build_consensus_data(data)

    bars = ax.bar(labels, values, color=colors_list,
                  edgecolor='white', linewidth=0.5, width=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    ax.axhline(1 / len(LABEL_ORDER), color='#888888', linestyle='--',
               linewidth=0.8, alpha=0.8, label='Prior uniforme')
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('Probabilidad', fontsize=10)
    ax.set_title(f'Consenso: {wc}', fontsize=10, fontweight='bold', pad=30)
    ax.legend(fontsize=8)


#Plot 3: Heatmap (solo clases con prob > EPSILON en algun modelo)
def plot_heatmap(ax, data):
    model_probs = data['model_probs']
    active = list(model_probs.keys())

    # Mostrar solo clases donde al menos un modelo supera el prior uniforme.
    # Los RF calibrados dan residuales pequeños a todas las clases,
    # por lo que el umbral util es 1/N_CLASSES, no un epsilon arbitrario.
    prior_threshold = 1.0 / len(LABEL_ORDER)
    EPSILON = prior_threshold
    visible_classes = [
        cl for cl in LABEL_ORDER
        if any(
            not np.isnan(model_probs[m].get(cl, np.nan))
            and model_probs[m].get(cl, 0.0) > EPSILON
            for m in active
        )
    ]
    if not visible_classes:
        visible_classes = LABEL_ORDER

    matrix = np.array([
        [model_probs[m].get(cl, np.nan) for cl in visible_classes]
        for m in active
    ])

    import matplotlib.colors as mcolors
    win_hex     = CLASS_COLORS.get(data['predicted'], '#d7191c')
    custom_cmap = mcolors.LinearSegmentedColormap.from_list(
        'win_cmap', ['#f7f7f7', win_hex], N=256
    )
    im = ax.imshow(matrix, aspect='auto', cmap=custom_cmap, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

    ax.set_xticks(range(len(visible_classes)))
    ax.set_xticklabels(visible_classes, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(active)))
    ax.set_yticklabels([MODEL_LABELS[m] for m in active], fontsize=9)

    for i in range(len(active)):
        for j in range(len(visible_classes)):
            val = matrix[i, j]
            if not np.isnan(val) and val > 0.05:
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=7.5,
                        color='white' if val > 0.55 else '#333333')

    if data['predicted'] in visible_classes:
        j_win = visible_classes.index(data['predicted'])
        ax.add_patch(plt.Rectangle(
            (j_win - 0.5, -0.5), 1, len(active),
            linewidth=1.5, edgecolor='#000000',
            facecolor='none', zorder=4,
        ))

    ax.set_title(f'Probabilidades por modelo y clase  (p > {EPSILON})',
                 fontsize=11, fontweight='bold', pad=10)


#Plot 4: Panel resumen
def plot_summary_panel(ax, data):
    ax.axis('off')
    margin  = data['margin']
    entropy = data['entropy']
    max_ent = np.log2(len(LABEL_ORDER))

    if margin > 0.4 and entropy < 1.0:
        conf_label, conf_color = 'ALTA',  '#1a9641'
    elif margin > 0.2 and entropy < 2.0:
        conf_label, conf_color = 'MEDIA', '#f4a736'
    else:
        conf_label, conf_color = 'BAJA',  '#d7191c'

    second_str = (f"{data['second']} ({data['second_prob']:.3f})"
                  if data['second'] else '-')

    divergence   = data.get('meta_divergence', 0.0)
    p_base_mean  = data.get('p_base_mean', data['max_prob'])
    show_div_warn = (USE_META
                     and divergence > DIV_THRESHOLD
                     and p_base_mean < BASE_MAX)
    div_color = '#d7191c' if show_div_warn else '#333333'

    lines = [
        ('OID',            data['oid'],                                  '#333333'),
        ('Clase predicha', data['predicted'],
                           CLASS_COLORS.get(data['predicted'], '#333333')),
        ('Prob. máxima',   f"{data['max_prob']:.4f}",                   '#333333'),
        ('Segunda clase',  second_str,                                   '#555555'),
        ('Margen (p1-p2)', f"{margin:.4f}",                             '#333333'),
        ('Entropía',       f"{entropy:.3f} / {max_ent:.3f} bits",       '#333333'),
        ('Modelos usados', f"{data['n_models']} / {len(ACTIVE_MODELS)}",  '#333333'),
        ('Confianza',      conf_label,                                    conf_color),
        ('Div. meta/base', f'{divergence:+.3f}',                          div_color),
    ]
    y_top = 0.96
    y_bot = 0.02
    dy    = (y_top - y_bot) / len(lines)
    y     = y_top
    for label, value, color in lines:
        val_str  = str(value)
        val_size = 7.5 if len(val_str) > 24 else 10
        ax.text(0.02, y, f'{label}:', transform=ax.transAxes,
                fontsize=10, color='#888888', va='top')
        ax.text(0.48, y, val_str, transform=ax.transAxes,
                fontsize=val_size, color=color, va='top', fontweight='bold')
        y -= dy

    title_str = 'Resumen  [⚠ DIVERGENCIA META/BASE]' if show_div_warn else 'Resumen'
    title_col = '#d7191c' if show_div_warn else 'black'
    ax.set_title(title_str, fontsize=11, fontweight='bold', pad=10, color=title_col)
    ax.add_patch(FancyBboxPatch(
        (0, 0), 1, 1, transform=ax.transAxes,
        boxstyle='round,pad=0.02', linewidth=1,
        edgecolor='#cccccc', facecolor='#f8f8f8', zorder=0,
    ))


#Figura completa
def plot_object(oid, df, save_path=None):
    if oid not in df.index:
        print(f"[ERROR] OID '{oid}' no encontrado en los resultados.")
        return

    data = get_object_data(oid, df)

    # Decidir tipo de axes para plot 2 antes de crear la figura (según cantidad de modelos, radar/barras)
    _, _, values_c, _ = _build_consensus_data(data)
    use_radar = len(values_c) >= 3

    fig = plt.figure(figsize=(18, 10))
    gs  = gridspec.GridSpec(
        2, 3, figure=fig,
        left=0.06, right=0.97, top=0.88, bottom=0.10,
        wspace=0.38, hspace=0.45,
    )

    ax1 = fig.add_subplot(gs[:, 0])
    plot_final_probs(ax1, data)

    ax2 = fig.add_subplot(gs[0, 1], polar=use_radar)
    pos = ax2.get_position()
    ax2.set_position([pos.x0, pos.y0 - 0.03, pos.width, pos.height])
    if use_radar:
        plot_model_consensus_radar(ax2, data)
    else:
        plot_model_consensus_bar(ax2, data)

    ax4 = fig.add_subplot(gs[0, 2])
    plot_summary_panel(ax4, data)

    ax3 = fig.add_subplot(gs[1, 1:])
    plot_heatmap(ax3, data)

    fig.suptitle(f'Clasificación: {oid}', fontsize=13, fontweight='bold', y=0.95)

    if save_path:
        import os
        os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
        print(f"Figura guardada -> {save_path}")

    plt.show()

print("Funciones de visualizacion cargadas.")


## 4. Explorador interactivo

In [ ]:
#Widget: selector de objeto
oid_input = widgets.Combobox(
    placeholder='Escribe o selecciona un OID...',
    options=list(results.index),
    ensure_option=False,
    layout=widgets.Layout(width='420px'),
    description='OID:',
    style={'description_width': '40px'},
)
btn_plot  = widgets.Button(description='Visualizar', button_style='primary', icon='bar-chart')
btn_prev  = widgets.Button(description='< Anterior', button_style='')
btn_next  = widgets.Button(description='Siguiente >', button_style='')
info_label = widgets.Label(value='')

#Guardado
save_dir = widgets.Text(
    value=SAVES_PATH,
    description='Carpeta:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '65px'},
)
fmt_label = widgets.Label(value='Formato:', layout=widgets.Layout(width='65px'))
save_fmt  = widgets.ToggleButtons(
    options=['png', 'pdf'],
    value='png',
    description='',
    layout=widgets.Layout(width='140px'),
    style={'button_width': '60px'},
)
btn_save    = widgets.Button(description='Guardar figura', button_style='warning', icon='save')
save_status = widgets.Label(value='')

out_widget = widgets.Output()
oid_list   = list(results.index)


def _current_idx():
    try:
        return oid_list.index(oid_input.value)
    except ValueError:
        return None


def on_plot(_):
    oid = oid_input.value.strip()
    with out_widget:
        clear_output(wait=True)
        if not oid:
            print("Introduce un OID.")
            return
        idx = _current_idx()
        if idx is not None:
            info_label.value = f'Objeto {idx + 1} / {len(oid_list)}'
        plot_object(oid, results, save_path=None)


def on_save(_):
    oid = oid_input.value.strip()
    if not oid:
        save_status.value = 'Sin OID seleccionado.'
        return
    import os
    os.makedirs(save_dir.value, exist_ok=True)
    path = os.path.join(save_dir.value, f'{oid}.{save_fmt.value}')
    plot_object(oid, results, save_path=path)
    save_status.value = f'Guardado: {path}'


def on_prev(_):
    idx = _current_idx()
    if idx is not None and idx > 0:
        oid_input.value = oid_list[idx - 1]
    on_plot(None)


def on_next(_):
    idx = _current_idx()
    if idx is not None and idx < len(oid_list) - 1:
        oid_input.value = oid_list[idx + 1]
    on_plot(None)


btn_plot.on_click(on_plot)
btn_save.on_click(on_save)
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)

display(widgets.VBox([
    widgets.HBox([oid_input, btn_plot]),
    widgets.HBox([btn_prev, btn_next, info_label]),
    widgets.HTML('<hr style="margin:6px 0; border-color:#ddd"/>'),
    widgets.HBox([save_dir]),
    widgets.HBox([fmt_label, save_fmt, btn_save, save_status],
                 layout=widgets.Layout(align_items='center')),
    out_widget,
]))


## 5. Panel por clase

In [ ]:
#Widget: panel por clase
# Muestra todos los objetos clasificados en una clase dada, con sus
# probabilidades finales y por modelo, ordenados por prob. descendente.
# Los objetos con esa clase como segunda opción aparecen diferenciados.

def build_class_table(df, target_class, top_n=None):
    """
    Construye un DataFrame con los objetos cuya clase predicha o segunda clase
    es target_class, con las columnas relevantes para esa clase.
    """
    # Columnas de probabilidad para esta clase
    final_col  = f'p_{target_class}'
    model_cols = [c for c in df.columns
                  if c.startswith(f'p_{target_class}_model')]

    # Objetos con target_class como clase predicha o segunda
    is_first  = df['predicted_class'] == target_class
    is_second = df.get('second_class', pd.Series(dtype=str)) == target_class

    mask = is_first | is_second
    sub  = df[mask].copy()

    if len(sub) == 0:
        return pd.DataFrame()

    # Columna de clasificación
    sub['clasificación'] = 'segunda'
    sub.loc[is_first[mask], 'clasificación'] = 'predicha'

    # Seleccionar y ordenar columnas
    entropy_model_cols = [c for c in df.columns
                          if c.startswith('entropy_model')]
    keep_cols = ['clasificación', final_col] + model_cols + ['entropy'] + entropy_model_cols
    keep_cols = [c for c in keep_cols if c in sub.columns]
    table = sub[keep_cols].sort_values(final_col, ascending=False)

    if top_n is not None and top_n > 0:
        table = table.head(top_n)

    return table


def style_class_table(table, target_class):
    """Aplica estilos visuales a la tabla: fondo por clasificación, gradiente de prob."""
    if table.empty:
        return None

    final_col  = f'p_{target_class}'
    model_cols = [c for c in table.columns
                  if c.startswith(f'p_{target_class}_model')]
    ent_model_cols = [c for c in table.columns if c.startswith('entropy_model')]
    prob_cols  = ([final_col] + model_cols
                  + (['entropy'] if 'entropy' in table.columns else [])
                  + ent_model_cols)

    import matplotlib.colors as _mc
    win_hex = CLASS_COLORS.get(target_class, '#377eb8')
    cmap    = _mc.LinearSegmentedColormap.from_list('win', ['#f7f7f7', win_hex], N=256)

    # Calcular color de texto (blanco/negro) según luminancia del fondo
    def _text_color(val):
        if pd.isna(val):
            return 'color: #333333'
        rgba   = cmap(float(val))
        # Luminancia relativa (ITU-R BT.709)
        lum    = 0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
        return 'color: white' if lum < 0.45 else 'color: #333333'

    def row_bg(row):
        if row['clasificación'] == 'predicha':
            return ['background-color: #f0f7ff; color: #333333'] * len(row)
        else:
            return ['background-color: #fff8f0; color: #333333'] * len(row)

    styled = (
        table.style
        .apply(row_bg, axis=1)
        .background_gradient(
            subset=[final_col],
            cmap=cmap,
            vmin=0, vmax=1,
        )
        .map(_text_color, subset=[final_col])
        .format({c: '{:.4f}' for c in prob_cols if c in table.columns})
        .set_table_styles([
            {'selector': 'th',
             'props': [('background-color', '#f2f2f2'),
                       ('font-size', '11px'),
                       ('text-align', 'center'),
                       ('color', '#333333')]},
            {'selector': 'td',
             'props': [('font-size', '11px'),
                       ('text-align', 'center'),
                       ('color', '#333333')]},
        ])
    )
    return styled


#Widgets
class_dropdown = widgets.Dropdown(
    options=LABEL_ORDER,
    value=LABEL_ORDER[0],
    description='Clase:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='200px'),
)
topn_input = widgets.BoundedIntText(
    value=50,
    min=1,
    max=10000,
    step=1,
    description='Top N:',
    style={'description_width': '50px'},
    layout=widgets.Layout(width='140px'),
)
topn_toggle = widgets.Checkbox(
    value=True,
    description='Limitar',
    layout=widgets.Layout(width='100px'),
)
btn_class   = widgets.Button(description='Mostrar tabla',
                             button_style='primary', icon='table')
btn_export  = widgets.Button(description='Exportar CSV',
                             button_style='warning', icon='download')
export_dir  = widgets.Text(
    value=SAVES_PATH,
    description='Carpeta:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '65px'},
)
export_status = widgets.Label(value='')
out_table     = widgets.Output()

_current_table = {}


def on_show_table(_):
    target = class_dropdown.value
    top_n  = topn_input.value if topn_toggle.value else None
    table  = build_class_table(results, target, top_n=top_n)
    _current_table['df']    = table
    _current_table['class'] = target

    with out_table:
        clear_output(wait=True)
        if table.empty:
            print(f"No hay objetos clasificados como '{target}' "
                  f"ni con '{target}' como segunda clase.")
            return

        n_first  = (table['clasificación'] == 'predicha').sum()
        n_second = (table['clasificación'] == 'segunda').sum()
        print(f"Clase '{target}':  "
              f"{n_first} objeto(s) con clase predicha  |  "
              f"{n_second} objeto(s) con segunda clase  "
              f"({'top ' + str(top_n) if top_n else 'todos'})")
        print()

        styled = style_class_table(table, target)
        if styled is not None:
            from IPython.display import display as ipy_display
            ipy_display(styled)


def on_export(_):
    table = _current_table.get('df', pd.DataFrame())
    target = _current_table.get('class', 'unknown')
    if table.empty:
        export_status.value = 'Tabla vacía — genera la tabla primero.'
        return
    import os
    os.makedirs(export_dir.value, exist_ok=True)
    path = os.path.join(export_dir.value, f'clase_{target}.csv')
    table.to_csv(path)
    export_status.value = f'Exportado'


btn_class.on_click(on_show_table)
btn_export.on_click(on_export)

display(widgets.VBox([
    widgets.HTML('<h3 style="margin-bottom:6px">5. Panel por clase</h3>'),
    widgets.HBox([class_dropdown, topn_toggle, topn_input, btn_class]),
    widgets.HTML('<hr style="margin:6px 0; border-color:#ddd"/>'),
    widgets.HBox([export_dir, btn_export, export_status]),
    out_table,
]))
